# 正解を判定できる課題で LLM を強化学習する (GRPO)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/24_llm_training/grpo.ipynb)

NLP クックブック 24-04 ／ LLM の学習と調整

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

算数の文章題のように正解を機械的に判定できる課題で、正解なら報酬を与える関数を書き、LLM が自分で生成した答えから強化学習 (GRPO) で学ばせたい。学習の前後で、学習に使っていない問題の正解率を比べる。

In [1]:
!pip install -q trl==1.18.0 peft==0.21.2 torchao==0.18.0   # Colab に入っている古い torchao (0.10) のままだと peft の import が失敗する

In [2]:
import json, re, time, unicodedata
import numpy as np
import pandas as pd
import sentencepiece as spm
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

set_seed(0)
name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"   # この章の「選好データで LLM を調整する (DPO)」と同じ 0.5B の指示チューニング済みモデル (MIT License)
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto")

# 学習用: SakanaAI/gsm8k-ja-test_250-1319 (Apache 2.0)。GSM8K（Cobbe et al., 2021、MIT License）のテストの 251 問目以降の
# 1,069 問を GPT-4 で日本語に訳したもの。先頭の 250 問は MGSM と重なるので、開発元があえて除いている
train_raw = load_dataset("json", data_files=hf_hub_download("SakanaAI/gsm8k-ja-test_250-1319", "gsm8k-ja-test_250-1319.jsonl",
                                                            repo_type="dataset"), split="train")
# 評価用: MGSM の日本語（juletxara/mgsm の ja）の 250 問全部。GSM8K のテストの先頭 250 問を人手で訳したもので、学習用の問題とは重ならない
# ライセンスは、元のリポジトリでは CC BY 4.0、データセットカードのメタデータでは CC BY-SA 4.0
test_raw = load_dataset("juletxara/mgsm", "ja", split="test")
print(len(train_raw), len(test_raw))
print(train_raw[1]["question"], "→", train_raw[1]["answer_number"])

1069 250
トラビスのタブレットには61個のアプリが入っていました。彼はもう使わない9個のアプリを削除し、さらに18個のアプリをダウンロードしました。現在、彼のタブレットには何個のアプリが入っていますか？ → 70


In [3]:
# 指示: 大規模言語モデルの章の Chain-of-Thought のレシピの「CoT (0-shot)」と同じ。途中の計算を書かせ、最後に答えを決まった形で書かせる
INSTRUCTION = "次の問題に答えてください。ステップごとに考えて途中の計算を書き、最後に「答えは〇〇です。」と書いてください。"
def to_prompt(question):
    return [{"role": "user", "content": f"{INSTRUCTION}\n\n問題: {question}"}]

def normalize(text):
    return unicodedata.normalize("NFKC", text).replace(",", "")

def final_answer(text):
    """決まった形（応答の最後の「答えは〇〇です。」）で書かれた答えを取り出す。形が違えば None"""
    m = re.search(r"答えは\s*[^\d\-\n]{0,5}?(-?\d+(?:\.\d+)?)[^\d\n]{0,10}$", normalize(text).strip())
    return float(m.group(1)) if m else None

def loose_answer(text):
    """形を問わず取り出す（大規模言語モデルの章の Chain-of-Thought のレシピの取り出し方の簡略版）: 「答え」の後ろ → 最後に出てきた数値"""
    text = normalize(text)
    found = re.findall(r"答え[はが]?\s*[:：]?[^\d\-\n]*(-?\d+(?:\.\d+)?)", text) or re.findall(r"-?\d+(?:\.\d+)?", text)
    return float(found[-1]) if found else None

# 報酬関数: GRPOTrainer は、生成した応答 (completions) と、データセットのほかの列（ここでは answer_number）をキーワード引数で渡す
def correct_reward(completions, answer_number, **kwargs):
    """決まった形で書いた答えが正解なら 1、それ以外は 0"""
    return [1.0 if final_answer(c[0]["content"]) == float(a) else 0.0 for c, a in zip(completions, answer_number)]

def format_reward(completions, **kwargs):
    """正誤に関係なく、応答が「答えは〇〇です。」の形で終わっていれば 0.2"""
    return [0.2 if final_answer(c[0]["content"]) is not None else 0.0 for c in completions]

examples = [[{"content": "3 + 4 = 7 なので、答えは7個です。"}], [{"content": "答えは7個です。よって 3 + 4 = 7"}], [{"content": "答えは8です。"}]]
print(correct_reward(examples, [7, 7, 7]), format_reward(examples))

[1.0, 0.0, 0.0] [0.2, 0.0, 0.2]


In [4]:
@torch.no_grad()
def evaluate(model, questions, max_new_tokens=320, bs=64):
    """貪欲法で解かせて、応答の文字列のリストと、生成した平均のトークン数を返す"""
    model.eval()
    tokenizer.padding_side = "left"
    outs, n_tok = [], []
    for i in range(0, len(questions), bs):
        enc = tokenizer.apply_chat_template([to_prompt(q) for q in questions[i:i + bs]], add_generation_prompt=True,
                                            return_tensors="pt", return_dict=True, padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        gen = out[:, enc["input_ids"].shape[1]:]
        n_tok += (gen != tokenizer.pad_token_id).sum(1).tolist()
        outs += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return outs, np.mean(n_tok)

def scores(outs, gold):
    gold = np.array(gold, dtype=float)
    strict = np.array([final_answer(o) == g for o, g in zip(outs, gold)])
    loose = np.array([loose_answer(o) == g for o, g in zip(outs, gold)])
    return strict, loose, np.mean([final_answer(o) is not None for o in outs])

gold = test_raw["answer_number"]
t = time.time()
before, before_len = evaluate(model, test_raw["question"])   # Colab の T4 で 1〜2 分
b_strict, b_loose, b_fmt = scores(before, gold)
print(f"学習前: 正解率（決まった形の答え）{b_strict.mean():.3f}、正解率（形を問わない）{b_loose.mean():.3f}、"
      f"形を守った割合 {b_fmt:.3f}、平均 {before_len:.0f} トークン（{time.time() - t:.0f} 秒）")
print("--- 学習前の出力の例:\n", before[0])

学習前: 正解率（決まった形の答え）0.072、正解率（形を問わない）0.252、形を守った割合 0.200、平均 192 トークン（34 秒）
--- 学習前の出力の例:
 ジャネットのアヒルは1日に16個の卵を生みますが、毎朝朝食の一環で3個を消費します。そのため、彼女に残っているのは16 - 3 = 13個の卵です。

次に、彼女は毎日4個を使って友達向けにマフィンを焼きます。つまり、1日あたり4個の卵が消費されます。

残りの卵は、1日あたり13 - 4 = 9個です。

ジャネットは市場でこれらの残り卵を1個あたり2ドルで売ります。したがって、彼女が市場で手に入れる金額は、9個 * 2ドル/個 = 18ドルです。

よって、答えは「彼女は毎日市場で18を得ています。」となります。


In [5]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

train_ds = Dataset.from_list([{"prompt": to_prompt(r["question"]), "answer_number": r["answer_number"]} for r in train_raw])
lora = LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
args = GRPOConfig(
    output_dir="grpo-out", max_steps=40, learning_rate=5e-5, warmup_steps=5,
    num_generations=8,                  # 1 問につき 8 個の応答を生成し、その中で報酬を比べる（グループ）
    per_device_train_batch_size=8, gradient_accumulation_steps=4,   # 1 ステップの更新に 32 応答 = 4 問 × 8 個
    steps_per_generation=8,             # 生成は 64 応答（8 問）をまとめて行い、2 ステップの更新に使う（生成の回数を減らして時間を短くする）
                                        # そのため 2 ステップ目の更新は、1 ステップ前のモデルが生成した応答を使う
    max_completion_length=320, temperature=1.0,
    logging_steps=5, save_strategy="no", report_to="none", seed=0,
    gradient_checkpointing=False, bf16=BF16, fp16=not BF16)   # GRPOConfig は bf16=True と gradient_checkpointing=True が既定なので明示する
trainer = GRPOTrainer(model=model, reward_funcs=[correct_reward, format_reward], args=args,
                      train_dataset=train_ds, processing_class=tokenizer, peft_config=lora)
trainer.model.print_trainable_parameters()
t = time.time()
trainer.train()   # Colab の T4 で 20〜25 分程度
print(f"学習 {time.time() - t:.0f} 秒、GPU メモリの最大使用量 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

trainable params: 10,076,160 || all params: 803,124,480 || trainable%: 1.2546


Step,Training Loss
5,0.019533
10,0.043929
15,0.030967
20,0.023957
25,0.059813
30,0.006671
35,0.049199
40,0.055162


学習 342 秒、GPU メモリの最大使用量 10.5 GB


In [6]:
# 報酬の推移（5 ステップごとの平均。1 ステップ = 4 問 × 8 個の応答）
logs = pd.DataFrame([l for l in trainer.state.log_history if "reward" in l])
cols = {"step": "ステップ", "rewards/correct_reward/mean": "正解の報酬", "rewards/format_reward/mean": "形式の報酬",
        "reward_std": "報酬の標準偏差", "completions/mean_length": "応答の長さ", "completions/clipped_ratio": "打ち切りの割合"}
print(logs[list(cols)].rename(columns=cols).round(3).to_string(index=False))

 ステップ  正解の報酬  形式の報酬  報酬の標準偏差   応答の長さ  打ち切りの割合
    5  0.057  0.043    0.275 229.682    0.224
   10  0.062  0.072    0.282 215.367    0.133
   15  0.068  0.088    0.296 220.031    0.146
   20  0.133  0.139    0.360 162.547    0.023
   25  0.240  0.170    0.430 163.906    0.016
   30  0.164  0.150    0.401 187.867    0.094
   35  0.307  0.171    0.467 175.484    0.068
   40  0.125  0.172    0.343 194.641    0.062


In [7]:
# 学習後に、同じ評価用の問題を解かせて比べる
model = trainer.model
t = time.time()
after, after_len = evaluate(model, test_raw["question"])   # Colab の T4 で 1〜2 分
a_strict, a_loose, a_fmt = scores(after, gold)
print(pd.DataFrame({"学習前": [b_strict.mean(), b_loose.mean(), b_fmt, before_len], "学習後": [a_strict.mean(), a_loose.mean(), a_fmt, after_len]},
                   index=["正解率（決まった形の答え）", "正解率（形を問わない）", "形を守った割合", "平均のトークン数"]).round(3).to_string())

def diff_ci(x, y, n_boot=2000):   # 同じ問題で比べる対応のあるブートストラップで、正解率の差の 95% 信頼区間
    idx = np.random.default_rng(0).integers(0, len(x), (n_boot, len(x)))
    d = x[idx].mean(1) - y[idx].mean(1)
    return f"{(x.mean() - y.mean()) * 100:+.1f} ポイント [{np.percentile(d, 2.5) * 100:+.1f}, {np.percentile(d, 97.5) * 100:+.1f}]"

print("学習後 − 学習前（決まった形の答え）:", diff_ci(a_strict, b_strict))
print("学習後 − 学習前（形を問わない）:", diff_ci(a_loose, b_loose))
print(f"GPU メモリの最大使用量 {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
# 問題ごとの入れ替わり（形を問わない正解で数える）
print(pd.crosstab(pd.Series(b_loose, name="学習前"), pd.Series(a_loose, name="学習後")).rename(index=str, columns=str).to_string())

# 出力の例: 学習前は形を守らずに正解していた問題と、学習前は正解で学習後に間違えた問題
for label, hits in [("形が直った", ~b_strict & b_loose & a_strict), ("学習後に間違えた", b_loose & ~a_loose)]:
    i = int(np.flatnonzero(hits)[0])
    print(f"\n=== {label}例: 問題 {i}（正解 {gold[i]:g}）\n[前] {before[i][-250:]}\n[後] {after[i][-250:]}")

                   学習前      学習後
正解率（決まった形の答え）    0.072    0.248
正解率（形を問わない）      0.252    0.252
形を守った割合          0.200    0.940
平均のトークン数       192.360  157.348
学習後 − 学習前（決まった形の答え）: +17.6 ポイント [+12.4, +22.8]
学習後 − 学習前（形を問わない）: +0.0 ポイント [-5.2, +4.8]
GPU メモリの最大使用量 10.5 GB
学習後    False  True
学習前               
False    165    22
True      22    41

=== 形が直った例: 問題 1（正解 3）
[前] まず、ローブを作成するために必要な青色の繊維の量は、青色の繊維が2巻分必要なので、単純に2巻分です。

次に、白色の繊維については、青色の繊維の半分を用いるとあります。したがって、白色の繊維は青色の繊維の半分なので、2巻分 / 2 = 1巻分となります。

最後に、必要な繊維の量をまとめると、青色の繊維が2巻分と白色の繊維が1巻分、合計で3巻分が必要となります。

よって、ローブを作成するためには3巻の繊維が必要です。

答え: 3巻
[後] ローブを作成するためには、青色の繊維を2巻分と白色の繊維をその半分用いる必要があります。まず、白色の繊維は青色の繊維の半分なので、白色の繊維は2巻分の半分である1巻分となります。

次に、青色の繊維を2巻分加えると、合計で3巻分が必要となります。

したがって、ローブを作成するには合計で3巻の繊維が必要です。答えは「3巻」です。

=== 学習後に間違えた例: 問題 0（正解 18）
[前] 日に16個の卵を生みますが、毎朝朝食の一環で3個を消費します。そのため、彼女に残っているのは16 - 3 = 13個の卵です。

次に、彼女は毎日4個を使って友達向けにマフィンを焼きます。つまり、1日あたり4個の卵が消費されます。

残りの卵は、1日あたり13 - 4 = 9個です。

ジャネットは市場でこれらの残り卵を1個あたり2ドルで売ります。したがって、彼女が市場で手に入れる金額は、9個 * 2ド

- GRPO (Group Relative Policy Optimization) は、Shao et al. (2024) が DeepSeekMath の論文で提案した強化学習の方法で、PPO の価値関数のモデルを使わず、同じ問題に対して生成した複数の応答（グループ、ここでは 8 個）の報酬の平均と標準偏差から、各応答がグループの中でどれだけ良かったか（アドバンテージ）を決める。DeepSeek-R1 (2025) は、数学の問題で、決まった形で書かせた最終的な答えの正誤と、形式を守ったかどうかを規則で判定する報酬を使った。ここも同じ考え方で、正解の報酬 (1) と形式の報酬 (0.2) を足している（このように規則で判定できる報酬は、検証できる報酬 (verifiable rewards) と呼ばれる）。グループの 8 個がすべて同じ報酬だとアドバンテージが 0 になって学習に使われないので、学習前に正解がほとんど出ない課題（正解率がほぼ 0）では、正解の報酬からは学べない（形式の報酬で差がつけば、形式だけは学ぶ）。
- この章の「選好データで LLM を調整する (DPO)」との違い: DPO は人が付けた選好の組（chosen と rejected）が要り、その応答は別のモデルの出力（方策外のデータ）だった。GRPO は選好の組も推論の過程の手本も要らず、問題と正解の数値と、それを判定する報酬関数だけで学習できる。応答は学習中のモデル自身がその場で生成する（方策上のデータ）。その代わり、学習のたびに生成するので時間がかかる。正解を機械的に判定できない課題（応答の役に立ち方など）には、判定役のモデル（報酬モデル）が要る。
- 上の出力では、正解率（決まった形の答え）は 0.072 → 0.248（+17.6 ポイント [+12.4, +22.8]）、形を守った割合は 0.200 → 0.940 に上がった。しかし形を問わない正解率は 0.252 → 0.252（+0.0 ポイント [−5.2, +4.8]）で、上がらなかった（信頼区間は 0 をまたぐ）。問題ごとに見ると、22 問が正解から不正解に、22 問が不正解から正解に変わった。この実行で身についたのは「決まった形で答える」ことで、40 ステップ（160 問）の学習では解く力は上がらなかった。報酬の推移でも、正解の報酬の上昇 (0.06〜0.07 → 0.13〜0.31) は形式の報酬の上昇 (0.04 → 0.17) と同時に起きている。1 ステップは 4 問だけなので、正解の報酬は問題の難しさで大きく揺れる。また、GPU の計算は実行ごとにわずかに変わり、それがサンプリングで生成する応答と学習の経過を変えるので、同じ種でも学習後の正解率は実行ごとに変わる（前の実行では、決まった形の答えの正解率が 0.216、形を問わない正解率が 0.216 だった）。1 回の実行で結論を出さず、種を変えて何回か学習して比べる。
- 応答は短くなった（評価で平均 192 → 157 トークン、学習中に上限の 320 トークンで打ち切られた割合は 0.22 → 0.02〜0.09）。打ち切られた応答は最後の「答えは〇〇です。」まで書けずに報酬が 0 になるので、短い応答が有利になると考えられる（短くなったことが正解率に影響したかどうかは確かめていない）。報酬が形式だけで取れるように設計すると、推論を書かずに答えだけを出すような、報酬は上がるが目的に合わない学習（報酬ハッキング）が起きることがある。報酬だけでなく、応答の長さと出力の例も見る。
- 大規模言語モデルの章の「考える過程を書かせて LLM に算数の文章題を解かせる（Chain-of-Thought と self-consistency）」との関係: 指示文はそのレシピの「CoT (0-shot)」と同じで、推論を書かせてから答えさせる。GRPO は推論の過程には報酬を与えず、最後の答えだけを判定する。推論の書き方は、正解に結びつくかどうかを通じて間接的に学ばれるだけで、途中の計算が正しいかどうかは確かめていない。上の「学習後に間違えた例」では、学習後の応答は「答えは〇〇です。」の形は守っているが、途中でマフィンに使った 4 個を引き忘れ、13 個 × 2 ドル = 26 ドルとしている（文の途中に英語も混じる）。そのレシピでは、形の違う答えも取り出せるように取り出し方を工夫したが、ここでは答えの形そのものを LoRA の重みに覚えさせた。正解の判定には、そのレシピの取り出し方より厳しい「決まった形」を使っている。取り出し方が緩いと、途中に出てきた数値がたまたま正解と一致しただけの応答にも報酬を与えてしまう。
- 時間とメモリ: 検証に使った GPU（T4 と同じ float16 の経路）では学習が 342 秒、全体で約 8 分、GPU メモリの最大使用量は 10.5GB だった（時間は GPU のほかの負荷で変わり、別の実行では学習が 736 秒、全体で約 17 分だった）。T4 では応答の生成が遅いので、この 2〜2.5 倍程度かかると見込んでいる（T4 では測っていない）。応答の生成は `use_vllm=True` で vLLM を使うと速くなるとされるが、Colab の torch などを入れ替えることになるので、ここでは使っていない（確かめていない）。`GRPOConfig` は trl 1.18.0 では `beta=0`（参照モデルとの KL の罰則なし、参照モデルを読み込まない）、学習率 1e-6、`bf16=True`、`gradient_checkpointing=True`、`loss_type="dapo"` が既定で、版によって変わることがある。
- データ: 学習用の SakanaAI/gsm8k-ja-test_250-1319 (Apache 2.0) は、データセットカードによると GPT-4 (gpt-4-0125-preview) で訳したもので、評価用の MGSM と重ならないように GSM8K のテストの 251 問目以降だけを使っている。どちらも GSM8K のテストの問題なので、ここで学習したモデルを GSM8K で評価すると、学習した問題で評価することになる。MGSM のライセンスと、評価データが事前学習に含まれている可能性については、大規模言語モデルの章の「考える過程を書かせて LLM に算数の文章題を解かせる（Chain-of-Thought と self-consistency）」の注意書きを参照。トークナイザーの修正については、大規模言語モデルの章の「オープンな LLM とチャットする（チャットテンプレート）」の注意書きを参照。